In [4]:
from typing import TypedDict, Optional
from enum import Enum

from langgraph.graph import StateGraph, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver


# ─────────────────────────────────────────────
# 1. Decision Enum
# ─────────────────────────────────────────────
class ReviewDecision(str, Enum):
    APPROVE = "approve"
    REJECT = "reject"
    ESCALATE = "escalate"


# ─────────────────────────────────────────────
# 2. State Definition
# ─────────────────────────────────────────────
class LoanState(TypedDict):
    application_id: str
    loan_amount: float
    applicant_summary: str
    risk_score: float

    ai_recommendation: Optional[str]
    review_required: bool
    human_decision: Optional[str]
    reviewer_id: Optional[str]
    review_notes: Optional[str]


# ─────────────────────────────────────────────
# 3. Business Rules
# ─────────────────────────────────────────────
HIGH_VALUE_THRESHOLD = 1_000_000
HIGH_RISK_THRESHOLD = 0.7


# ─────────────────────────────────────────────
# 4. ROUTE NODE (must return state)
# ─────────────────────────────────────────────
def route_node(state: LoanState):
    return state


# ─────────────────────────────────────────────
# 5. ROUTER FUNCTION (returns path)
# ─────────────────────────────────────────────
def decide_route(state: LoanState):
    if (
        state["loan_amount"] >= HIGH_VALUE_THRESHOLD
        or state["risk_score"] >= HIGH_RISK_THRESHOLD
    ):
        return "human_review"
    return "auto_approve"


# ─────────────────────────────────────────────
# 6. AUTO APPROVAL NODE
# ─────────────────────────────────────────────
def auto_approve(state: LoanState):
    state["ai_recommendation"] = "Auto-approved by system (low risk)"
    state["review_required"] = False
    return state


# ─────────────────────────────────────────────
# 7. HUMAN-IN-THE-LOOP NODE
# ─────────────────────────────────────────────
def human_review(state: LoanState):
    decision = interrupt({
        "message": "Human review required for loan application",
        "application_id": state["application_id"],
        "loan_amount": state["loan_amount"],
        "risk_score": state["risk_score"]
    })

    state["human_decision"] = decision["decision"]
    state["reviewer_id"] = decision.get("reviewer_id", "UNKNOWN")
    state["review_notes"] = decision.get("notes", "")

    return state


# ─────────────────────────────────────────────
# 8. FINAL NODE
# ─────────────────────────────────────────────
def finalize(state: LoanState):
    if state.get("human_decision") == "approve":
        state["ai_recommendation"] = "Approved by human reviewer"
    elif state.get("human_decision") == "reject":
        state["ai_recommendation"] = "Rejected by human reviewer"
    else:
        state["ai_recommendation"] = "Escalated for further review"

    return state


# ─────────────────────────────────────────────
# 9. BUILD GRAPH
# ─────────────────────────────────────────────
graph = StateGraph(LoanState)

graph.add_node("route", route_node)
graph.add_node("auto_approve", auto_approve)
graph.add_node("human_review", human_review)
graph.add_node("finalize", finalize)

graph.set_entry_point("route")


# ─────────────────────────────────────────────
# 10. ROUTING LOGIC
# ─────────────────────────────────────────────
graph.add_conditional_edges(
    "route",
    decide_route,
    {
        "auto_approve": "auto_approve",
        "human_review": "human_review",
    },
)

graph.add_edge("auto_approve", "finalize")
graph.add_edge("human_review", "finalize")
graph.add_edge("finalize", END)


# ─────────────────────────────────────────────
# 11. CHECKPOINTER (IMPORTANT FIX)
# ─────────────────────────────────────────────
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)


# ─────────────────────────────────────────────
# 12. TEST INPUT
# ─────────────────────────────────────────────
loan_input = {
    "application_id": "APP100302",
    "loan_amount": 1500000,
    "applicant_summary": "Self-employed, credit score 660, DTI 45%",
    "risk_score": 0.62,
    "ai_recommendation": None,
    "review_required": False,
    "human_decision": None,
    "reviewer_id": None,
    "review_notes": None,
}


# ─────────────────────────────────────────────
# 13. THREAD ID (IMPORTANT)
# ─────────────────────────────────────────────
config = {"configurable": {"thread_id": "loan-app-001"}}


# ─────────────────────────────────────────────
# 14. RUN WORKFLOW (FIRST STEP)
# ─────────────────────────────────────────────
result = app.invoke(loan_input, config=config)

print("\nFINAL RESULT (or paused state):")
print(result)


FINAL RESULT (or paused state):
{'application_id': 'APP100302', 'loan_amount': 1500000, 'applicant_summary': 'Self-employed, credit score 660, DTI 45%', 'risk_score': 0.62, 'ai_recommendation': None, 'review_required': False, 'human_decision': None, 'reviewer_id': None, 'review_notes': None, '__interrupt__': [Interrupt(value={'message': 'Human review required for loan application', 'application_id': 'APP100302', 'loan_amount': 1500000, 'risk_score': 0.62}, id='793cff4f5f0ea453b039f5fbe5be1f6e')]}


In [5]:
from langgraph.types import Command

result = app.invoke(
    Command(
        resume={
            "decision": "approve",
            "reviewer_id": "UNDERWRITER_001",
            "notes": "Income verified, stable employment confirmed"
        }
    ),
    config={"configurable": {"thread_id": "loan-app-001"}}
)

print("\nRESUMED FINAL RESULT:")
print(result)


RESUMED FINAL RESULT:
{'application_id': 'APP100302', 'loan_amount': 1500000, 'applicant_summary': 'Self-employed, credit score 660, DTI 45%', 'risk_score': 0.62, 'ai_recommendation': 'Approved by human reviewer', 'review_required': False, 'human_decision': 'approve', 'reviewer_id': 'UNDERWRITER_001', 'review_notes': 'Income verified, stable employment confirmed'}
